In [77]:
from sklearn.datasets import load_iris

from sklearn.model_selection import train_test_split, cross_validate

from sklearn.naive_bayes import MultinomialNB
from sklearn.metrics import accuracy_score, auc, confusion_matrix, f1_score, recall_score
from sklearn.svm import SVC

In [78]:
from collections import Counter
from nltk.util import ngrams

import numpy as np

In [79]:
from pathlib import Path

In [80]:
from textfabric_utils import get_verses
from eval_utils import predict, predict_proba, metricise, find_mislabels, find_top_k_words, get_top_n_grams, split_list
import fitting_utils

In [81]:
# Try Fs("book@en").items() to get names of the book included in the text-fabric dataset
ot_train_books = {
    "Genesis": list(range(1,50+1)),
    "Exodus": list(range(1,21+1))
}

ot_test_books = {
    "Deuteronomy": list(range(1,20+1))
}

In [82]:
# Parse the dataset and get verses
# each verse in a tuple "(`verse reference`: str, `transliteration as a list of words`: list[str])"

ot_train_book_verses = get_verses(ot_train_books)
# print(ot_train_book_verses)

ot_test_book_verses = get_verses(ot_test_books)

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,65,6566.69,100
chapter,1269,336.36,100
verse,31341,13.62,100
word,426835,1.00,100


Genesis
Exodus


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,65,6566.69,100
chapter,1269,336.36,100
verse,31341,13.62,100
word,426835,1.00,100


Deuteronomy


In [83]:
nt_train_books = {
    "Matthew": list(range(1,30+1)),
    "Mark": list(range(1,16+1)),
    "Luke": list(range(1,24+1)),
    "John": list(range(1,21+1)),
}

nt_test_books = {
    "Acts": list(range(1,28+1))
}

In [84]:
nt_train_book_verses = get_verses(nt_train_books, target_fabric="etcbc/syrnt", ver="0.1")
# print(nt_train_book_verses)

nt_test_book_verses = get_verses(nt_test_books, target_fabric="etcbc/syrnt", ver="0.1")

**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,27,4060.74,100
chapter,260,421.69,100
lexeme,3038,36.09,100
verse,7957,13.78,100
word,109640,1.00,100


Matthew
Mark
Luke
John


**Locating corpus resources ...**

Name,# of nodes,# slots / node,% coverage
book,27,4060.74,100
chapter,260,421.69,100
lexeme,3038,36.09,100
verse,7957,13.78,100
word,109640,1.00,100


Acts


## Create a list of all verses for training

In [85]:
train_data_dict = ot_train_book_verses.copy()
train_data_dict.update(nt_train_book_verses)

ot_train_verses = [vrs for verses in ot_train_book_verses.values() for vrs in verses]
nt_train_verses = [vrs for verses in nt_train_book_verses.values() for vrs in verses]

print(len(ot_train_verses))
print(len(nt_train_verses))

train_verse_txts = ot_train_verses.copy()
train_verse_txts.extend(nt_train_verses)

train_verse_labels = [0 for i in range(len(ot_train_verses))]
train_verse_labels.extend([1 for i in range(len(nt_train_verses))])
print(len(train_verse_labels))

2115
3779
5894


# Generate Bag-of-Words counters & define model's vocabulary

In [10]:
char_n_gram_vocabs = fitting_utils.make_char_n_gram_vocab(train_verse_txts)
print(f"Found {len(char_n_gram_vocabs[0])} independent n-grams from {len(char_n_gram_vocabs[1])} verses!")

Total words parsed: 78918
Found 7449 independent n-grams from 5894 verses!


In [11]:
# char_n_grams = char_n_gram_vocabs[0]
# print(char_n_grams)
print(len(char_n_gram_vocabs[0].keys()))

7449


# Vectorise the verses

In [12]:
# Count the number of each n-gram per verse,
# reusing the counters created while constructing the vocabulary

assert(len(char_n_gram_vocabs[1]) == len(train_verse_txts))

In [13]:
char_n_gram_feat = fitting_utils.make_feature(char_n_gram_vocabs)

In [14]:
print(len(char_n_gram_feat))

5894


## Train classifier

In [15]:
# n_gram_train, n_gram_test, n_gram_train_y, n_gram_test_y = train_test_split(n_gram_feats, labels, test_size=0.2, random_state=0)

# gnb = GaussianNB()

# c_clf = gnb.fit(char_n_gram_feat, train_verse_labels)

# print(f"{train_verse_labels[0]}, {train_verse_labels[-1]}")

In [16]:
mnb = MultinomialNB()

c_mnb = mnb.fit(char_n_gram_feat, train_verse_labels)

print(f"{train_verse_labels[0]}, {train_verse_labels[-1]}")

0, 1


In [17]:
# cnb = ComplementNB()

# c_cnb = cnb.fit(char_n_gram_feat, train_verse_labels)

# print(f"{train_verse_labels[0]}, {train_verse_labels[-1]}")

## Prepare test data

### OT

In [18]:
ot_test_verses = [vrs for book in ot_test_book_verses.values() for vrs in book]

# Labels
ot_test_y_np = np.empty(len(ot_test_verses), dtype=int)
ot_test_y_np.fill(0)
print(ot_test_y_np.shape)

# Vectorise with Bag-of-Words counting
ot_test_X = fitting_utils.vectorise(ot_test_verses, char_n_gram_vocabs[0], ' '.join)
ot_test_X_np = np.array(ot_test_X)

(555,)
Parsed 8348 words from 555 verses


### NT

In [19]:
nt_test_verses = [vrs for book in nt_test_book_verses.values() for vrs in book]

# Labels
nt_test_y_np = np.empty(len(nt_test_verses), dtype=int)
nt_test_y_np.fill(1)
print(nt_test_y_np.shape)

# Features
nt_test_X = fitting_utils.vectorise(nt_test_verses, char_n_gram_vocabs[0], ' '.join)
nt_test_X_np = np.array(nt_test_X)

(1007,)
Parsed 15383 words from 1007 verses


In [105]:
all_test_verses = []
all_test_verses.extend(ot_test_verses)
all_test_verses.extend(nt_test_verses)

all_test_y = []
all_test_y.extend(ot_test_y_np)
all_test_y.extend(nt_test_y_np)

## Evaluate

In [21]:
ot_y_proba, ot_y_pred, num_fp, num_tn = predict_proba(c_mnb, ot_test_X_np, ot_test_y_np)

Number of mislabeled points out of a total 555 verses: 96
Local accuracy: 0.83


In [22]:
nt_y_proba, nt_y_pred, num_fn, num_tp = predict_proba(c_mnb, nt_test_X_np, nt_test_y_np)

Number of mislabeled points out of a total 1007 verses: 23
Local accuracy: 0.98


In [23]:
metricise(all_test_y, ot_y_pred, nt_y_pred)

Overall Accuracy: 0.92
Overall Recall: 0.98
F1 Score: 0.94


(0.9238156209987196, 0.9771598808341608, 0.9429803545759463)

## Format data for manual inspection

In [24]:
def csvify(key: str, verses_d: dict[str, list[tuple[str, list[str]]]]) -> str:
    """Convert the verse data into a CSV-formatted string."""
    # Set header line
    result = '"Verse Reference No.","Probability","No. Words","ܐܠܦܒܝܬ ܣܘܪܝܝܐ","ETCBC Transliteration"\n'
    # Extract & format verse data
    verses = verses_d[key]
    for verse in verses:
        line = f'"{verse[0]}",{verse[1]:.02f},{len(verse[2])},"{' '.join(verse[2])}","{' '.join(verse[3])}"\n'
        result += line
    return result

In [25]:
ot_mislabels = {
    "Deuteronomy": []
}

nt_mislabels = {
    "Acts": []
}

pos = 0

# OT
for i in range(len(ot_test_verses)):
    if ot_test_y_np[i] != ot_y_pred[i]:
        res = (
            ot_test_verses[i][0],
            ot_y_proba[i][0],
            ot_test_verses[i][2],
            ot_test_verses[i][1]
        )
        ot_mislabels["Deuteronomy"].append(res)


# NT
for i in range(len(nt_y_pred)):
    if nt_test_y_np[i] != nt_y_pred[i]:
        res = (
            nt_test_verses[i][0],
            nt_y_proba[i][1],
            nt_test_verses[i][2],
            nt_test_verses[i][1]
        )
        nt_mislabels["Acts"].append(res)

In [26]:
print(len(ot_mislabels["Deuteronomy"]))

96


In [27]:
Path(f"./out/prediction_data_ot.csv").write_text(csvify("Deuteronomy", ot_mislabels))
Path(f"./out/prediction_data_nt.csv").write_text(csvify("Acts", nt_mislabels))

3873

### Create a subset of training data with proper nouns removed

In [112]:
def remove_proper_nouns(verses: list) -> list:
    """Create a copy of verses with pre-defined proper nouns removed."""
    result_verses = verses.copy()
    FREQUENT_PROPER_NOUN = {"JCW<", "MWC>", "J<QWB", ">JSRJL", "JWSP", ">BRHM", "CM<WN", "LJCW"}
    for i in range(len(verses)):
        if len(set(verses[i][1]) & FREQUENT_PROPER_NOUN) > 0:
            excludes = []
            verses_range = range(len(verses[i][1]))

            for j in verses_range:
                if verses[i][1][j] in FREQUENT_PROPER_NOUN:
                    excludes.append(j)
            
            translit_r = []
            syriac_r = []
            removed = []
            
            for j in verses_range:
                if j not in excludes:
                    translit_r.append(verses[i][1][j])
                    syriac_r.append(verses[i][2][j])
                else:
                    removed.append(verses[i][1][j])
            print(f"Removed {removed} from verses[{i}] ({verses[i][0]})")
            result_verses[i] = (verses[i][0], translit_r, syriac_r)
        return result_verses

In [113]:
char_n = 3

train_verses_removed = remove_proper_nouns(train_verses)

char_n_gram_vocabs_removed = fitting_utils.make_char_n_gram_vocab(train_verses_removed)
print(f"Found {len(char_n_gram_vocabs_removed[0])} independent n-grams from {len(char_n_gram_vocabs_removed[1])} verses!")

char_n_gram_feat_removed = fitting_utils.make_feature(char_n_gram_vocabs_removed)

mnb = MultinomialNB()
c_mnb = mnb.fit(char_n_gram_feat_removed, train_verse_labels)

print("--> OT")
ot_test_X_c = fitting_utils.vectorise(ot_test_verses, char_n_gram_vocabs_removed[0], ' '.join, span=char_n)
ot_test_X_c_np = np.array(ot_test_X_c)
ot_y_proba, ot_y_pred, num_fn, num_tp = predict_proba(c_mnb, ot_test_X_c_np, ot_test_y_np)

print("--> NT")
nt_test_X_c = fitting_utils.vectorise(nt_test_verses, char_n_gram_vocabs_removed[0], ' '.join, span=char_n)
nt_test_X_c_np = np.array(nt_test_X_c)
nt_y_proba, nt_y_pred, num_fn, num_tp = predict_proba(c_mnb, nt_test_X_c_np, nt_test_y_np)

print("--> ALL")
metrics = metricise(all_test_y, ot_y_pred, nt_y_pred)
print(metrics)

Total words parsed: 77581
Found 7453 independent n-grams from 5894 verses!
--> OT
Parsed 8348 words from 555 verses
Number of mislabeled points out of a total 555 verses: 403
Local accuracy: 0.27
--> NT
Parsed 15383 words from 1007 verses
Number of mislabeled points out of a total 1007 verses: 254
Local accuracy: 0.75
--> ALL
Overall Accuracy: 0.58
Overall Recall: 0.75
F1 Score: 0.70
(0.5793854033290653, 0.7477656405163853, 0.6962552011095701)


In [115]:
ot_predictions = {
    "Deuteronomy": []
}

nt_predictions = {
    "Acts": []
}

pos = 0

# OT
for i in range(len(ot_test_verses)):
    res = (
        ot_test_verses[i][0],
        ot_y_proba[i][0],
        ot_test_verses[i][2],
        ot_test_verses[i][1]
    )
    ot_predictions["Deuteronomy"].append(res)


# NT
for i in range(len(nt_y_pred)):
    res = (
        nt_test_verses[i][0],
        nt_y_proba[i][1],
        nt_test_verses[i][2],
        nt_test_verses[i][1]
    )
    nt_predictions["Acts"].append(res)

print(len(ot_predictions["Deuteronomy"]))

Path(f"./out/prediction_results_removed_ot.csv").write_text(csvify("Deuteronomy", ot_predictions))
Path(f"./out/prediction_results_removed_nt.csv").write_text(csvify("Acts", nt_predictions))

555


201626

#### Count words

In [28]:
find_top_k_words(train_verse_txts)

[1623 1352 1081 1053  877  875  831  687  672  664]
['ܡܢ' 'ܠܗ' 'ܕܝܢ' 'ܠܐ' 'ܗܘܐ' 'ܐܢܐ' 'ܗܘ' 'ܠܗܘܢ' 'ܐܡܪ' 'ܥܠ']
['MN' 'LH' 'DJN' 'L>' 'HW>' '>N>' 'HW' 'LHWN' '>MR' '<L']


(array(['ܡܢ', 'ܠܗ', 'ܕܝܢ', 'ܠܐ', 'ܗܘܐ', 'ܐܢܐ', 'ܗܘ', 'ܠܗܘܢ', 'ܐܡܪ', 'ܥܠ',
        'ܝܫܘܥ', 'ܘܐܡܪ', 'ܗܘܘ', 'ܐܢܬܘܢ', 'ܠܟܘܢ', 'ܡܛܠ', 'ܘܠܐ', 'ܘܐܡ̣ܪ',
        'ܟܕ', 'ܐܢܬ', 'ܓܝܪ', 'ܡܪܝܐ', 'ܗܢܐ', 'ܐܢܘܢ', 'ܕܠܐ', 'ܠܝ', 'ܐܦ',
        'ܐܠܐ', 'ܐܝܟ', 'ܐܠܗܐ', 'ܗܠܝܢ', 'ܠܟ', 'ܚܕ', 'ܘܟܕ', 'ܐܪܥܐ', 'ܠܘܬ',
        'ܗܝ', 'ܡܕܡ', 'ܗܐ', 'ܟܠ', 'ܐܢ', 'ܡܢܐ', 'ܒܗ', 'ܗܕܐ', 'ܘܡܢ', 'ܕܐܠܗܐ',
        'ܩܕܡ', 'ܒܪ', 'ܗܢܘܢ', 'ܐܝܬ', 'ܥܡ', 'ܐܬܐ', 'ܐܘ', 'ܘܐܡܪܝܢ', 'ܒܝܬ',
        'ܗܘܬ', 'ܥܕܡܐ', 'ܓܒܪܐ', 'ܬܡܢ', 'ܐܝܟܢܐ', 'ܥܢܐ', 'ܒܪܗ', 'ܐܡܪܝܢ',
        'ܐܢܫ', 'ܡܘܫܐ', 'ܗܟܢܐ', 'ܝܥܩܘܒ', 'ܐܝܣܪܝܠ', 'ܠܢ', 'ܟܠܗܘܢ', 'ܢܗܘܐ',
        'ܦܪܥܘܢ', 'ܒܐܪܥܐ', 'ܝܘܣܦ', 'ܬܘܒ', 'ܕܐܝܬ', 'ܣܓܝܐܐ', 'ܡܢܗ', 'ܟܠܗ',
        'ܐܢܬܬܐ', 'ܥܡܗ', 'ܥܡܐ', 'ܥܠܘܗܝ', 'ܘܥܠ', 'ܬܠܡܝܕܘܗܝ', 'ܡܐ', 'ܘܐܢ',
        'ܗܝܕܝܢ', 'ܕܡܢ', 'ܐܢܫܐ', 'ܡܪܝ', 'ܐܒܪܗܡ', 'ܝܕܥ', 'ܘܗܐ', 'ܠܘܬܗ',
        'ܘܐܡܪܘ', 'ܐܡܝܢ', 'ܕܡܨܪܝܢ', 'ܪܒܐ', 'ܘܐܦ', 'ܟܢܫܐ', 'ܐܒܝ', 'ܗܟܝܠ',
        'ܕܐܢܫܐ', 'ܒܬܪ', 'ܥܒܕ', 'ܚܢܢ', 'ܐܝܠܝܢ', 'ܟܗܢܐ', 'ܫ̈ܢܝܢ', 'ܡܢܘ',
        'ܡܫܟܚ', 'ܟܐܦܐ', 'ܒܪܐ', 'ܗܫܐ', 'ܪܘܚܐ', 'ܕܐܢܐ', 'ܘܟܠ', 'ܫܡܝܐ

In [29]:
# OT
find_top_k_words(ot_train_verses)

[605 453 338 303 297 271 257 219 213 212]
['ܡܢ' 'ܘܐܡ̣ܪ' 'ܡܪܝܐ' 'ܠܗ' 'ܥܠ' 'ܡܛܠ' 'ܠܐ' 'ܐܠܗܐ' 'ܐܢܐ' 'ܐܪܥܐ']
['MN' 'W>M#R' 'MRJ>' 'LH' '<L' 'MVL' 'L>' '>LH>' '>N>' '>R<>']


(array(['ܡܢ', 'ܘܐܡ̣ܪ', 'ܡܪܝܐ', 'ܠܗ', 'ܥܠ', 'ܡܛܠ', 'ܠܐ', 'ܐܠܗܐ', 'ܐܢܐ',
        'ܐܪܥܐ', 'ܗܢܐ', 'ܠܗܘܢ', 'ܐܢܘܢ', 'ܘܠܐ', 'ܠܝ', 'ܐܝܟ', 'ܦܪܥܘܢ',
        'ܐܝܣܪܝܠ', 'ܝܥܩܘܒ', 'ܩܕܡ', 'ܗܐ', 'ܠܟ', 'ܐܢܬ', 'ܡܘܫܐ', 'ܝܘܣܦ',
        'ܒܐܪܥܐ', 'ܠܘܬ', 'ܟܠ', 'ܗܘܐ', 'ܕܡܨܪܝܢ', 'ܐܦ', 'ܫ̈ܢܝܢ', 'ܒܝܬ',
        'ܐܒܪܗܡ', 'ܘܗܘ̣ܐ', 'ܒܢ̈ܝ', 'ܗܠܝܢ', 'ܕܠܐ', 'ܠܟܘܢ', 'ܒܪ', 'ܟܕ', 'ܥܡ',
        'ܗ̣ܘ', 'ܘܥܠ', 'ܥܡܐ', 'ܠܡܘܫܐ', 'ܗܟܢܐ', 'ܘܡܢ', 'ܡ̈ܝܐ', 'ܕܐܪܥܐ',
        'ܘܗܐ', 'ܘܐܡ̣ܪܬ', 'ܒܪܐ', 'ܐܢ', 'ܘܩ̣ܪܐ', 'ܗܝ', 'ܐܡ̇ܪ', 'ܪܒܐ', 'ܥܡܗ',
        'ܬܡܢ', 'ܒܬܪ', 'ܘܐܡܪܘ', 'ܥܣܘ', 'ܓܒܪܐ', 'ܘܐܦ', 'ܥܕܡܐ', 'ܘܚ̣ܙܐ',
        'ܐܢܬܬܐ', 'ܛܒ', 'ܫܡܗ', 'ܚܕ', 'ܐܝܣܚܩ', 'ܠܡܪܝܐ', 'ܟܠܗܘܢ', 'ܘܟܕ', 'ܗܘ',
        'ܠܐܪܥܐ', 'ܒܗ', 'ܗܕܐ', 'ܘܟܠ', 'ܠܗ̇', 'ܕܦܪܥܘܢ', 'ܕܐܝܬ', 'ܠܢ',
        'ܠܝܥܩܘܒ', 'ܟܠܗ̇', 'ܗܘܘ', 'ܘܐܢ', 'ܐܒܘܗܝ', 'ܡܕܡ', 'ܕܟܢܥܢ', 'ܐܢܬܘܢ',
        'ܒܝܘܡܐ', 'ܡܢܐ', 'ܠܥܡܐ', 'ܥܢܐ', 'ܕܡܪܝܐ', 'ܠܦܪܥܘܢ', 'ܢܗܘܐ', 'ܐܢܬܬܗ',
        'ܬܘܒ', 'ܗ̇ܘ', 'ܬܡ̇ܢ', 'ܡ̇ܠܟܐ', 'ܠܒܢ', 'ܘܐܘܠܕ', 'ܘܗܠܝܢ', 'ܝܘܡ̈ܝܢ',
        'ܐܒܪܡ', 'ܐܚܘܗܝ', 'ܐܡ̣ܪ', 'ܠܐܒܪܗܡ', 'ܒܪܬ', 'ܟܠܗ', 'ܘܥܕܡܐ', 

In [30]:
find_top_k_words(nt_train_verses)

[1075 1049 1018  796  783  783  668  662  632  542]
['ܕܝܢ' 'ܠܗ' 'ܡܢ' 'ܠܐ' 'ܗܘܐ' 'ܗܘ' 'ܐܡܪ' 'ܐܢܐ' 'ܝܫܘܥ' 'ܠܗܘܢ']
['DJN' 'LH' 'MN' 'L>' 'HW>' 'HW' '>MR' '>N>' 'JCW<' 'LHWN']


(array(['ܕܝܢ', 'ܠܗ', 'ܡܢ', 'ܠܐ', 'ܗܘܐ', 'ܗܘ', 'ܐܡܪ', 'ܐܢܐ', 'ܝܫܘܥ', 'ܠܗܘܢ',
        'ܘܐܡܪ', 'ܗܘܘ', 'ܐܢܬܘܢ', 'ܠܟܘܢ', 'ܓܝܪ', 'ܥܠ', 'ܟܕ', 'ܘܠܐ', 'ܐܢܬ',
        'ܐܠܐ', 'ܕܠܐ', 'ܐܦ', 'ܗܠܝܢ', 'ܚܕ', 'ܗܢܐ', 'ܘܟܕ', 'ܠܝ', 'ܐܢܘܢ',
        'ܡܕܡ', 'ܡܛܠ', 'ܗܢܘܢ', 'ܗܝ', 'ܐܝܟ', 'ܡܢܐ', 'ܠܟ', 'ܘܐܡܪܝܢ', 'ܕܐܠܗܐ',
        'ܐܬܐ', 'ܐܢ', 'ܒܗ', 'ܐܝܬ', 'ܗܕܐ', 'ܠܘܬ', 'ܐܡܪܝܢ', 'ܐܘ', 'ܗܘܬ',
        'ܘܡܢ', 'ܐܝܟܢܐ', 'ܐܢܫ', 'ܟܠ', 'ܣܓܝܐܐ', 'ܒܪܗ', 'ܒܪ', 'ܬܠܡܝܕܘܗܝ',
        'ܥܢܐ', 'ܗܐ', 'ܥܡ', 'ܝܕܥ', 'ܥܕܡܐ', 'ܓܒܪܐ', 'ܗܝܕܝܢ', 'ܐܡܝܢ', 'ܢܗܘܐ',
        'ܟܢܫܐ', 'ܕܡܢ', 'ܬܡܢ', 'ܡܢܗ', 'ܗܟܝܠ', 'ܠܢ', 'ܥܠܘܗܝ', 'ܥܒܕ', 'ܟܗܢܐ',
        'ܕܐܢܫܐ', 'ܐܢܫܐ', 'ܐܝܠܝܢ', 'ܬܘܒ', 'ܡܐ', 'ܟܠܗܘܢ', 'ܟܠܗ', 'ܒܝܬ',
        'ܐܠܗܐ', 'ܠܘܬܗ', 'ܡܫܟܚ', 'ܡܪܝ', 'ܟܐܦܐ', 'ܕܐܝܬ', 'ܪܘܚܐ', 'ܗܟܢܐ',
        'ܩܕܡ', 'ܐܙܠ', 'ܐܒܝ', 'ܡܢܘ', 'ܚܙܐ', 'ܫܡܥܘܢ', 'ܘܐܬܐ', 'ܕܐܢܐ', 'ܕܐܡܪ',
        'ܚܕܐ', 'ܪܒܝ', 'ܚܢܢ', 'ܢܒܝܐ', 'ܘܐܢ', 'ܐܢܬܬܐ', 'ܠܝܫܘܥ', 'ܝܕܥܝܢ',
        'ܠܝܬ', 'ܡܠܬܐ', 'ܬܪܝܢ', 'ܗܫܐ', 'ܡܢܗܘܢ', 'ܢܦܫܗ', 'ܕܐܬܐ', 'ܫܡܝܐ',
        'ܐܚܪܢܐ', 'ܠܚܡܐ', 'ܥܡܗ', 'ܗܘܝ', 'ܝܘܚܢܢ', 'ܡܠܟܐ', 

### Most common n-grams

In [31]:
syr_n_gram_vocabs = fitting_utils.make_char_n_gram_vocab(train_verse_txts, mode=2)

get_top_n_grams(char_n_gram_vocabs[0], syr_vocabs=syr_n_gram_vocabs[0])

Total words parsed: 78918
[4973 4874 4158 3428 3386 2913 2871 2722 2705 2499]
[['W' 'N' ' ']
 ['J' 'N' ' ']
 ['>' ' ' 'D']
 ['N' '>' ' ']
 ['>' ' ' 'W']
 ['L' '>' ' ']
 ['T' '>' ' ']
 [' ' '>' 'N']
 ['>' ' ' '>']
 ['>' ' ' 'L']]


(array([['ܘ', 'ܢ', ' '],
        ['ܝ', 'ܢ', ' '],
        ['ܐ', ' ', 'ܕ'],
        ['ܢ', 'ܐ', ' '],
        ['ܐ', ' ', 'ܘ'],
        ['ܠ', 'ܐ', ' '],
        ['ܬ', 'ܐ', ' '],
        [' ', 'ܐ', 'ܢ'],
        ['ܐ', ' ', 'ܐ'],
        ['ܐ', ' ', 'ܠ'],
        [' ', 'ܗ', 'ܘ'],
        [' ', 'ܘ', 'ܐ'],
        [' ', 'ܠ', 'ܗ'],
        ['ܡ', 'ܢ', ' '],
        [' ', 'ܡ', 'ܢ'],
        ['ܝ', 'ܐ', ' '],
        ['ܐ', 'ܡ', 'ܪ'],
        ['ܢ', ' ', 'ܐ'],
        ['ܪ', 'ܐ', ' '],
        [' ', 'ܕ', 'ܐ'],
        ['ܐ', ' ', 'ܡ'],
        ['ܗ', 'ܘ', 'ܢ'],
        ['ܗ', 'ܝ', ' '],
        [' ', 'ܠ', 'ܐ'],
        [' ', 'ܕ', 'ܝ'],
        ['ܢ', ' ', 'ܕ'],
        ['ܡ', 'ܐ', ' '],
        ['ܐ', ' ', 'ܗ'],
        ['ܠ', 'ܗ', ' '],
        ['ܕ', 'ܝ', 'ܢ'],
        ['ܘ', 'ܗ', 'ܝ'],
        ['ܘ', 'ܐ', 'ܡ'],
        ['ܢ', ' ', 'ܠ'],
        ['ܡ', 'ܪ', ' '],
        ['ܢ', ' ', 'ܘ'],
        ['ܘ', 'ܐ', ' '],
        [' ', 'ܐ', 'ܝ'],
        ['ܐ', ' ', 'ܒ'],
        ['ܗ', 'ܘ', 'ܐ'],
        ['ܪ', ' ', 'ܠ'],


In [32]:
# Word-level intertextuality (What words appear in both texts)

## Try different n of character n-grams

### Cross-validation

In [33]:
print(len(train_verse_txts))

for i in range(len(train_verse_txts)):
    if len(train_verse_txts[i]) != 3 or not isinstance(train_verse_txts[i], tuple):
        print(i)
        print(train_verse_txts[i])

5894


In [34]:
fold = 5
n_window = 1

npgen = np.random.default_rng(seed=100)

shuffled_ids = np.arange(len(train_verse_txts), dtype=int)
npgen.shuffle(shuffled_ids)
print(shuffled_ids)
# Shuffle lists with a naïve approach since train_verse_txts cannot be converted to np.array
train_verses = []
for idx in shuffled_ids:
    train_verses.append(train_verse_txts[idx])
train_labels = np.array(train_verse_labels)[shuffled_ids]

# Split the training set into {fold} chunks
train_sample_splits = split_list(train_verses, parts=fold)
train_label_splits = split_list(train_labels, parts=fold)
print(len(train_sample_splits))

[ 757 3096 2901 ...  964 1028 2968]
NOTE: the number of training samples (5894) is not divisible by 5.
Resulting split of sub-arrays will be uneven.
NOTE: the number of training samples (5894) is not divisible by 5.
Resulting split of sub-arrays will be uneven.
5


In [35]:
# Cross-Validation
for hold_out in range(fold):
    est = fitting_utils.BoW_Estimator(MultinomialNB(), ' '.join, n=n_window)
    train_samples = []
    train_classes = []
    vald_samples = []
    vald_classes = []
    accuracies = []
    recalls = []
    f_ones = []
    for i in range(len(train_sample_splits)):
        if i == hold_out:
            vald_samples = train_sample_splits[i]
            vald_classes = train_label_splits[i]
        else:
            train_samples.extend(train_sample_splits[i])
            train_classes.extend(train_label_splits[i])
    # Fit and evaluate the estimator on this subset
    est.fit(train_samples, train_classes)
    current_pred = est.predict(vald_samples)
    acc, rca, f_one = metricise(vald_classes, y_all=current_pred)
    accuracies.append(acc)
    recalls.append(rca)
    f_ones.append(f_one)
    hold_out += 1

print("==== Result ====")
print(f"{fold}-fold Cross-Validation")
print(f"<Accuracy> avg: {np.average(accuracies):.02f}, std: {np.std(accuracies):.02f}")
print(f"<Recall> avg: {np.average(recalls):.02f}, std: {np.std(recalls):.02f}")
print(f"<F1 Score> avg: {np.average(f_ones):.02f}, std: {np.std(f_ones):.02f}")

Total words parsed: 63348
Parsed 15528 words from 1177 verses
Overall Accuracy: 0.98
Overall Recall: 0.99
F1 Score: 0.98
Total words parsed: 63005
Parsed 15871 words from 1178 verses
Overall Accuracy: 0.99
Overall Recall: 1.00
F1 Score: 0.99
Total words parsed: 63270
Parsed 15606 words from 1178 verses
Overall Accuracy: 0.98
Overall Recall: 1.00
F1 Score: 0.98
Total words parsed: 63017
Parsed 15859 words from 1178 verses
Overall Accuracy: 0.99
Overall Recall: 1.00
F1 Score: 0.99
Total words parsed: 62864
Parsed 16012 words from 1180 verses
Overall Accuracy: 0.98
Overall Recall: 1.00
F1 Score: 0.99
==== Result ====
5-fold Cross-Validation
<Accuracy> avg: 0.98, std: 0.00
<Recall> avg: 1.00, std: 0.00
<F1 Score> avg: 0.99, std: 0.00


In [36]:
for char_n in range(1,6):
    print(f"\nchar_n = {char_n}")
    char_n_gram_vocabs = fitting_utils.make_char_n_gram_vocab(train_verse_txts, span=char_n)
    print(f"Found {len(char_n_gram_vocabs[0])} independent n-grams from {len(char_n_gram_vocabs[1])} verses!")

    if char_n== 1:
        # print(fitting_utils(char_n_gram_vocabs[0])
        # get_top_n_grams(char_n_gram_vocabs[0])
        ot_vocab = fitting_utils.make_char_n_gram_vocab(ot_train_verses, span=1)
        nt_vocab = fitting_utils.make_char_n_gram_vocab(nt_train_verses, span=1)
        print("OT")
        ot_counts = get_top_n_grams(ot_vocab[0])
        print("NT")
        nt_counts = get_top_n_grams(nt_vocab[0])
        print(ot_counts)
        print(nt_counts)

    # if char_n == 2 or char_n == 4:
        # print(list(char_n_gram_vocabs[0].keys())[0:11])
    
    char_n_gram_feat = fitting_utils.make_feature(char_n_gram_vocabs)
    
    mnb = MultinomialNB()
    c_mnb = mnb.fit(char_n_gram_feat, train_verse_labels)
    
    print("--> OT")
    ot_test_X_c = fitting_utils.vectorise(ot_test_verses, char_n_gram_vocabs[0], ' '.join, span=char_n)
    ot_test_X_c_np = np.array(ot_test_X_c)
    ot_y_proba, ot_y_pred, num_fn, num_tp = predict_proba(c_mnb, ot_test_X_c_np, ot_test_y_np)
    
    print("--> NT")
    nt_test_X_c = fitting_utils.vectorise(nt_test_verses, char_n_gram_vocabs[0], ' '.join, span=char_n)
    nt_test_X_c_np = np.array(nt_test_X_c)
    nt_y_proba, nt_y_pred, num_fn, num_tp = predict_proba(c_mnb, nt_test_X_c_np, nt_test_y_np)
    
    print("--> ALL")
    metrics = metricise(all_test_y, ot_y_pred, nt_y_pred)
    print(metrics)


char_n = 1
Total words parsed: 78918
Found 27 independent n-grams from 5894 verses!
Total words parsed: 28504
Total words parsed: 50414
OT
[26389 16150 13479  9795  9214  8988  8332  7685  6469  5990]
[[' ']
 ['>']
 ['W']
 ['J']
 ['N']
 ['L']
 ['M']
 ['R']
 ['B']
 ['H']]
NT
[46635 28531 21443 20100 18586 15333 13530 12590 11879 10328]
[[' ']
 ['>']
 ['W']
 ['N']
 ['J']
 ['L']
 ['M']
 ['D']
 ['H']
 ['T']]
(None, array([[' '],
       ['>'],
       ['W'],
       ['J'],
       ['N'],
       ['L'],
       ['M'],
       ['R'],
       ['B'],
       ['H'],
       ['T'],
       ['D'],
       ['<'],
       ['K'],
       ['"'],
       ['#'],
       ['C'],
       ['X'],
       ['Q'],
       ['P'],
       ['^'],
       ['S'],
       ['V'],
       ['G'],
       ['Z'],
       ['Y']], dtype='<U1'), array([26389, 16150, 13479,  9795,  9214,  8988,  8332,  7685,  6469,
        5990,  5946,  5901,  4427,  3448,  2935,  2831,  2758,  2607,
        2257,  2054,  2039,  1707,  1049,   841,   781,   567]))


## Word n-grams with various values for n

### Try a specific one

In [519]:
word_n = 1  # n of word n-grams

In [520]:
word_n_gram_vocabs = make_word_n_gram_vocab(train_verse_txts, span=word_n)
print(f"Found {len(word_n_gram_vocabs[0])} independent n-grams from {len(word_n_gram_vocabs[1])} verses!")

Total words parsed: 78289
Found 14178 independent n-grams from 5841 verses!


In [521]:
word_n_gram_feat = make_feature(word_n_gram_vocabs)
print(len(word_n_gram_feat[0]))
print(len(word_n_gram_feat))

14178
5841


In [522]:
mnb = MultinomialNB()

w_mnb = mnb.fit(word_n_gram_feat, train_verse_labels)

print(f"{train_verse_labels[0]}, {train_verse_labels[-1]}")

0, 1


In [523]:
# Vectorise with Bag-of-Words counting
ot_test_X_w = vectorise(ot_test_verses, make_bow(word_n_gram_vocabs[0]), identity, span=word_n)
ot_test_X_w_np = np.array(ot_test_X_w)

Parsed 8348 words from 555 verses


In [524]:
ot_y_proba, ot_y_pred, num_fn, num_tp = predict_proba(w_mnb, ot_test_X_w_np, ot_test_y_np)

Number of mislabeled points out of a total 555 verses: 555
Local accuracy: 0.00


In [525]:
# Features
nt_test_X_w = vectorise(nt_test_verses, make_bow(word_n_gram_vocabs[0]), identity, span=word_n)
nt_test_X_w_np = np.array(nt_test_X_w)

Parsed 15383 words from 1007 verses


In [526]:
nt_y_proba, nt_y_pred, num_fn, num_tp = predict_proba(w_mnb, nt_test_X_w_np, nt_test_y_np)

Number of mislabeled points out of a total 1007 verses: 0
Local accuracy: 1.00


In [527]:
metricise(all_test_y, ot_y_pred, nt_y_pred)

Overall Accuracy: 0.92
Overall Recall: 1.00
F1 Score: 0.78


(0.6446862996158771, 1.0, 0.7839626313740755)

### Test out several n's

In [166]:
for word_n in range(1,6):
    print(f"\nword_n = {word_n}")
    word_n_gram_vocabs = make_word_n_gram_vocab(train_verse_txts, span=word_n)
    print(f"Found {len(word_n_gram_vocabs[0])} independent n-grams from {len(word_n_gram_vocabs[1])} verses!")
    
    word_n_gram_feat = make_feature(word_n_gram_vocabs)
    
    mnb = MultinomialNB()
    w_mnb = mnb.fit(word_n_gram_feat, train_verse_labels)
    
    print("OT")
    ot_test_X_w = vectorise(ot_test_verses, make_bow(word_n_gram_vocabs[0]), identity, span=word_n)
    ot_test_X_w_np = np.array(ot_test_X_w)
    ot_y_proba, ot_y_pred, num_fn, num_tp = predict_proba(w_mnb, ot_test_X_w_np, ot_test_y_np)
    
    print("NT")
    nt_test_X_w = vectorise(nt_test_verses, make_bow(word_n_gram_vocabs[0]), identity, span=word_n)
    nt_test_X_w_np = np.array(nt_test_X_w)
    nt_y_proba, nt_y_pred, num_fn, num_tp = predict_proba(w_mnb, nt_test_X_w_np, nt_test_y_np)

    print("ALL")
    metricise(all_test_y, ot_y_pred, nt_y_pred)


word_n = 1
Total words parsed: 78918
Found 14238 independent n-grams from 5894 verses!
OT
Parsed 8348 words from 555 verses
Number of mislabeled points out of a total 555 verses: 145
Local accuracy: 0.74
NT
Parsed 15383 words from 1007 verses
Number of mislabeled points out of a total 1007 verses: 30
Local accuracy: 0.97
ALL
Overall Accuracy: 0.89
Overall Recall: 0.97
F1 Score: 0.92

word_n = 2
Total words parsed: 78918
Found 50229 independent n-grams from 5894 verses!
OT
Parsed 8348 words from 555 verses
Number of mislabeled points out of a total 555 verses: 224
Local accuracy: 0.60
NT
Parsed 15383 words from 1007 verses
Number of mislabeled points out of a total 1007 verses: 56
Local accuracy: 0.94
ALL
Overall Accuracy: 0.82
Overall Recall: 0.94
F1 Score: 0.87

word_n = 3
Total words parsed: 78918
Found 59442 independent n-grams from 5894 verses!
OT
Parsed 8348 words from 555 verses
Number of mislabeled points out of a total 555 verses: 457
Local accuracy: 0.18
NT
Parsed 15383 words

## SVM

### Character N-grams

In [416]:
svc = SVC()

csvc_clf = svc.fit(char_n_gram_feat, train_verse_labels)

# OT
ot_y_pred, num_fp, num_tn = predict(csvc_clf, ot_test_X_np, ot_test_y_np)

# NT
nt_y_pred, num_fn, num_tp = predict(csvc_clf, nt_test_X_np, nt_test_y_np)

metricise(all_test_y, ot_y_pred, nt_y_pred)

Number of mislabeled points out of a total 555 OT verses: 142
Local accuracy: 0.74
Number of mislabeled points out of a total 1007 OT verses: 23
Local accuracy: 0.98
Overall Precision: 0.87
Overall Recall: 0.98
F1 Score: 0.92


### Word N-grams

In [422]:
import time

svc = SVC()

print(time.asctime())
wsvc_clf = svc.fit(word_n_gram_feat, train_verse_labels)

print(time.asctime())
print("fitted!")

Sat Mar 22 14:16:19 2025
Sat Mar 22 14:21:19 2025
fitted!


In [423]:
# OT
ot_y_pred, num_fp, num_tn = predict(wsvc_clf, ot_test_X_w_np, ot_test_y_np)

# NT
nt_y_pred, num_fn, num_tp = predict(wsvc_clf, nt_test_X_w_np, nt_test_y_np)

metricise(all_test_y, ot_y_pred, nt_y_pred)

Number of mislabeled points out of a total 555 OT verses: 548
Local accuracy: 0.01
Number of mislabeled points out of a total 1007 OT verses: 0
Local accuracy: 1.00
Overall Precision: 0.65
Overall Recall: 1.00
F1 Score: 0.79


## Word counts in the dataset

In [425]:
# Average no. words (with std) per verse, in training and test sets, also by books (possibly).

all_wc = []
ot_wc = []
nt_wc = []
all_cc = []
nt_cc = []
ot_cc = []

for book in ot_train_book_verses.keys():
    print(f"Book: {book}")
    book_wc = []
    book_cc = []
    verses = ot_train_book_verses[book]
    for ot_verse in verses:
        # print(ot_verse)
        wc = len(ot_verse[1])
        c_cnt = [len(word) for word in ot_verse]
        ot_wc.append(wc)
        all_wc.append(wc)
        book_wc.append(wc)
        all_cc.extend(c_cnt)
        ot_cc.extend(c_cnt)
        book_cc.extend(c_cnt)
    print(f"Number of verses: {len(book_wc)}")
    print(f"Avg word count: {np.average(book_wc)}")
    print(f"Word count std: {np.std(book_wc)}\n")
    print(f"Avg char count: {np.average(book_cc)}")
    print(f"Char count std: {np.std(book_cc)}")

for book in nt_train_book_verses.keys():
    print(f"Book: {book}")
    book_wc = []
    book_cc = []
    verses = nt_train_book_verses[book]
    for nt_verse in verses:
        wc = len(nt_verse[1])
        c_cnt = [len(word) for word in ot_verse]
        nt_wc.append(wc)
        all_wc.append(wc)
        book_wc.append(wc)
        all_cc.extend(c_cnt)
        nt_cc.extend(c_cnt)
        book_cc.extend(c_cnt)
    print(f"Number of verses: {len(book_wc)}")
    print(f"Avg word count: {np.average(book_wc)}")
    print(f"Word count std: {np.std(book_wc)}\n")
    print(f"Avg char count: {np.average(book_cc)}")
    print(f"Char count std: {np.std(book_cc)}")

Book: Genesis
Number of verses: 1533
Avg word count: 13.09849967384214
Word count std: 4.8717489401625675

Avg char count: 17.583387692976736
Char count std: 7.494576872678524
Book: Exodus
Number of verses: 582
Avg word count: 14.474226804123711
Word count std: 5.5310919676008075

Avg char count: 18.06758304696449
Char count std: 6.809972860104494
Book: Matthew
Number of verses: 1071
Avg word count: 13.052287581699346
Word count std: 4.985232336843567

Avg char count: 22.666666666666668
Char count std: 2.3570226039551585
Book: Mark
Number of verses: 678
Avg word count: 12.969026548672566
Word count std: 4.831428201103894

Avg char count: 22.666666666666668
Char count std: 2.3570226039551585
Book: Luke
Number of verses: 1098
Avg word count: 13.301457194899818
Word count std: 4.995919798628114

Avg char count: 22.666666666666668
Char count std: 2.357022603955158
Book: John
Number of verses: 879
Avg word count: 14.1160409556314
Word count std: 5.495283304087178

Avg char count: 22.6666666

In [426]:
# Overall train verses statistics
print(f"Avg word count across all train verses: {np.average(all_wc)}\n")

# OT train verses statistics
print(f"OT train verses: {len(ot_wc)}")
print(f"Avg word count: {np.average(ot_wc)}")
print(f"Avg char count: {np.average(ot_cc)}")
print(f"Word count std: {np.std(ot_wc)}")
print(f"Char count std: {np.std(ot_cc)}\n")

# NT train verses statistics
print(f"NT train verses: {len(nt_wc)}")
print(f"Avg word count: {np.average(nt_wc)}")
print(f"Avg char count: {np.average(nt_cc)}")
print(f"Word count std: {np.std(nt_wc)}")
print(f"Char count std: {np.std(nt_cc)}\n")

Avg word count across all train verses: 13.403355589796268

OT train verses: 2115
Avg word count: 13.477068557919623
Avg char count: 17.716627265563435
Word count std: 5.098909994539582
Char count std: 7.3157805817189105

NT train verses: 3726
Avg word count: 13.361513687600644
Avg char count: 22.666666666666668
Word count std: 5.105017399286848
Char count std: 2.3570226039551585



In [428]:
all_test_wc = []
ot_test_wc = []
nt_test_wc = []

for book in ot_test_book_verses.keys():
    print(f"Book: {book}")
    book_test_wc = []
    verses = ot_test_book_verses[book]
    for ot_verse in verses:
        # print(ot_verse)
        wc = len(ot_verse[1])
        # print(wc)
        ot_test_wc.append(wc)
        all_test_wc.append(wc)
        book_test_wc.append(wc)
    print(f"Number of verses: {len(book_test_wc)}")
    print(f"Avg word count: {np.average(book_test_wc)}")
    print(f"Std: {np.std(book_test_wc)}\n")

for book in nt_test_book_verses.keys():
    print(f"Book: {book}")
    book_test_wc = []
    verses = nt_test_book_verses[book]
    for nt_verse in verses:
        wc = len(nt_verse[1])
        nt_test_wc.append(wc)
        all_test_wc.append(wc)
        book_test_wc.append(wc)
    print(f"Number of verses: {len(book_test_wc)}")
    print(f"Avg word count: {np.average(book_test_wc)}")
    print(f"Std: {np.std(book_test_wc)}\n")

# Overall test verses statistics
print(f"Avg word count across all words: {np.average(all_wc)}\n")

Book: Deuteronomy
Number of verses: 555
Avg word count: 15.04144144144144
Std: 5.780371702697472

Book: Acts
Number of verses: 1007
Avg word count: 15.276067527308838
Std: 5.572600478658499

Avg word count across all words: 13.403355589796268

